In [2]:
import numpy as np
import matplotlib.pyplot as plt
from lm_polygraph.ue_metrics.pred_rej_area import PredictionRejectionArea
from lm_polygraph.ue_metrics.ue_metric import (
    get_random_scores,
    normalize_metric,
)
import sklearn
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from collections import defaultdict
from sacrebleu import CHRF, BLEU
from lm_polygraph.utils.manager import UEManager

from lm_polygraph.ue_metrics import PredictionRejectionArea

metrics = [PredictionRejectionArea(max_rejection=0.5)]

In [3]:
model='llama'
prefix=''
dataset='xsum'
manager = UEManager.load(f'processed_mans/{model}{prefix}_{dataset}_train_full_enriched.man')

/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f89e420d960>]


In [4]:
manager.gen_metrics.keys()

dict_keys([('sequence', 'Rouge_rouge1'), ('sequence', 'Rouge_rouge2'), ('sequence', 'Rouge_rougeL'), ('sequence', 'BLEU'), ('sequence', 'Bert'), ('sequence', 'Sbert'), ('sequence', 'Accuracy'), ('sequence', 'AlignScore'), ('sequence', 'AlignScoreInputOutput')])

In [5]:
def load_managers(dataset, model='llama', model_type='base', suff ='test_qe_enriched'):
    prefix = '' if model_type == 'base' else '_instruct'
    if model =='llama':
        # manager = UEManager.load(f'processed_mans/{model}{prefix}_{dataset}_test_qe_enriched.man')

        train_manager =  UEManager.load(f'mans/{model}{prefix}_{dataset}_train_{suff}.man') if suff!='' else  UEManager.load(f'mans/{model}{prefix}_{dataset}_train.man') 
        train_manager.metrics = metrics
        train_manager.eval_ue()
    else:
        # manager = UEManager.load(f'processed_mans/{model}{prefix}_{dataset}_test_processed.man')

        train_manager =  UEManager.load(f'mans/{model}{prefix}_{dataset}_train_{suff}.man') if suff!='' else  UEManager.load(f'mans/{model}{prefix}_{dataset}_train.man') 

    return train_manager

def extract_and_prepare_data_hello(dataset, methods_dict, all_metrics, model='llama', model_type='base', suff ='test_qe_enriched'):
    train_manager = load_managers(dataset, model, model_type, suff=suff)

    full_ue_methods = list(methods_dict.keys())
    ue_methods = list(methods_dict.values())

    # sequences = manager.stats['greedy_tokens']
    # texts = manager.stats['greedy_texts']
    # targets = manager.stats['target_texts']

    train_sequences = train_manager.stats['greedy_tokens']
    train_texts = train_manager.stats['greedy_texts']
    train_targets = train_manager.stats['target_texts']

    train_gen_lengths = np.array([len(seq) for seq in train_sequences])
    # gen_lengths = np.array([len(seq) for seq in sequences])

    # Get train and test values for metrics and UE, remove union of nans
    test_nans = []
    train_nans = []

    train_metric_values = {}
    test_metric_values = {}
    for metric in all_metrics:
        # values = np.array(manager.gen_metrics[('sequence', metric)])
        # test_metric_values[metric] = np.array(values)
        # test_nans.extend(np.argwhere(np.isnan(values)).flatten())

        train_values = np.array(train_manager.gen_metrics[('sequence', metric)])
        train_metric_values[metric] = np.array(train_values)
        train_nans.extend(np.argwhere(np.isnan(train_values)).flatten())

    train_ue_values = {}
    # test_ue_values = {}
    for i, method in enumerate(full_ue_methods):
        train_values = np.array(train_manager.estimations[('sequence', method)])
        train_ue_values[ue_methods[i]] = train_values
        train_nans.extend(np.argwhere(np.isnan(train_values)).flatten())

        # values = np.array(manager.estimations[('sequence', method)])
        # test_ue_values[ue_methods[i]] = values
        # test_nans.extend(np.argwhere(np.isnan(values)).flatten())

    train_nans = np.unique(train_nans).astype(int)
    # test_nans = np.unique(test_nans).astype(int)

    # Remove nans
    for metric in all_metrics:
        # test_metric_values[metric] = np.delete(test_metric_values[metric], test_nans)
        train_metric_values[metric] = np.delete(train_metric_values[metric], train_nans)

    for method in ue_methods:
        # test_ue_values[method] = np.delete(test_ue_values[method], test_nans)
        train_ue_values[method] = np.delete(train_ue_values[method], train_nans)

    train_gen_lengths = np.delete(train_gen_lengths, train_nans)

    return train_ue_values, train_metric_values, train_gen_lengths


In [36]:
methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
}

DATASETS = [
    'wmt14_deen',
    'wmt14_fren',
    'wmt14_csen',
    'wmt14_ruen',
    'wmt19_ruen',
    'wmt19_fien',
    'wmt19_deen',
    'wmt19_lten'
]

all_metrics = ['Comet', 'XComet-XCOMET-XXL', 'metricx-metricx-24-hybrid-large-v2p6']
all_methods =['MSP', 'PPL', 'MTE', 'MCSE', 'MCNSE']

In [6]:
methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
}

DATASETS = [
    'xsum',
]

all_metrics = ['AlignScoreInputOutput', 'Rouge_rougeL']
all_methods =['MSP', 'PPL', 'MTE', 'MCSE', 'MCNSE']

In [7]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import linregress
import numpy as np
import pandas as pd

sns.set(style="whitegrid", font_scale=1.4, rc={"font.family": "serif"})

methrics_dict ={
  'Comet':'Comet', 'XComet-XCOMET-XXL' :'XComet-XXL', 'metricx-metricx-24-hybrid-large-v2p6' :'MetricX-Large' ,
  'AlignScoreInputOutput':'Align Score', 'Accuracy':'Acc', 'AlignScoreInputOutput':'Align Score','Rouge_rougeL':'Rouge L'
}

def format_dataset_name(raw_name):
    try:
        prefix, lang_pair = raw_name.split("_")
        prefix = prefix.upper()

        if len(lang_pair) == 4:  # e.g., fren → Fr-En
            src = lang_pair[:2].capitalize()
            tgt = lang_pair[2:].capitalize()
            lang_fmt = f"{src}-{tgt}"
        else:
            lang_fmt = lang_pair.upper()

        return f"{prefix} {lang_fmt}"
    except Exception:
        return raw_name.upper()


def plot_metric_vs_length(
    gen_lengths, metric_values,
    metric_name, dataset_name,
    mode='Train', save_path='plot.pdf', model=''
):
    from sklearn.linear_model import LinearRegression
    from scipy.stats import linregress

    # Trim outliers
    upper_q, lower_q = np.quantile(gen_lengths, [0.95, 0.05])
    mask = (gen_lengths > lower_q) & (gen_lengths < upper_q)
    gen_lengths = gen_lengths[mask]
    metric_values = metric_values[mask]

    # Normalize
    scaler_len = MinMaxScaler()
    scaler_val = MinMaxScaler()

    norm_len = scaler_len.fit_transform(gen_lengths[:, None]).squeeze()
    norm_val = scaler_val.fit_transform(metric_values[:, None]).squeeze()

    # Bin and smooth
    df = pd.DataFrame({"length": norm_len, "metric": norm_val})
    grouped = df.groupby("length").agg(['mean', 'sem'])
    x_vals = grouped.index.values
    y_vals = grouped['metric']['mean'].values
    y_errs = grouped['metric']['sem'].values

    # Fit regression (on raw normalized data)
    linreg = LinearRegression().fit(norm_len[:, None], norm_val)
    slope = linreg.coef_[0]

    # Also compute p-value
    slope_, intercept_, r_val, p_val, std_err = linregress(norm_len, norm_val)

    x_line = np.linspace(0, 1, 100)
    y_line = linreg.predict(x_line[:, None])

    # Plot
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(x_vals, y_vals, label='AVG metric value', color="navy")
    ax.fill_between(x_vals, y_vals - y_errs, y_vals + y_errs, alpha=0.2, color="navy")
    ax.plot(x_line, y_line, linestyle='--', color='crimson', label='Regression Line')

    ax.text(0.05, 0.95,
            f"Slope: {slope:.2f}\n$p$-value: {p_val:.3f}",
            transform=ax.transAxes, fontsize=12, verticalalignment='top',
            bbox=dict(boxstyle="round,pad=0.3", facecolor='white', edgecolor='gray'))

    pretty_dataset = format_dataset_name(dataset_name)
    ax.set_title(f"{methrics_dict[metric_name]} vs. Length ({pretty_dataset})", fontsize=14)
    ax.set_xlabel("Generated sequence length (normalized)")
    ax.set_ylabel(f"{methrics_dict[metric_name]} (normalized)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.show()


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import os 
DATASETS =['xsum']
all_metrics = ['AlignScoreInputOutput', 'Rouge_rougeL']
models =['llama','gemma']
for model in models:

    for dataset in DATASETS:
        train_ue_values, train_metric_values, train_gen_lengths = extract_and_prepare_data_hello(dataset, methods_dict, all_metrics, model=model, suff='full_enriched')

        for metric in all_metrics:
            os.makedirs(f'{model}/{metric}', exist_ok=True)

            plot_metric_vs_length(
                gen_lengths=np.array(train_gen_lengths),
                metric_values=np.array(train_metric_values[metric]),
                metric_name=metric,
                dataset_name=dataset,
                mode='Train',
                save_path=f'{model}/{metric}/{dataset}_{metric}_{model}_train.pdf',
                # model = model
            )

        # plot_metric_vs_length(
        #     gen_lengths=np.array(test_gen_lengths),
        #     metric_values=np.array(test_metric_values[metric]),
        #     metric_name=metric,
        #     dataset_name=dataset,
        #     mode='Test',
        #     save_path=f'{dataset}_{metric}_test.pdf'
        # )


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import os 
DATASETS =['gsm8k']
all_metrics=['Accuracy']
models =['llama','gemma']
for model in models:

    for dataset in DATASETS:
        train_ue_values, train_metric_values, train_gen_lengths = extract_and_prepare_data_hello(dataset, methods_dict, all_metrics, model=model, suff='full_enriched')

        for metric in all_metrics:
            os.makedirs(f'{model}/{metric}', exist_ok=True)

            plot_metric_vs_length(
                gen_lengths=np.array(train_gen_lengths),
                metric_values=np.array(train_metric_values[metric]),
                metric_name=metric,
                dataset_name=dataset,
                mode='Train',
                save_path=f'{model}/{metric}/{dataset}_{metric}_{model}_train.pdf',
                # model = model
            )

        # plot_metric_vs_length(
        #     gen_lengths=np.array(test_gen_lengths),
        #     metric_values=np.array(test_metric_values[metric]),
        #     metric_name=metric,
        #     dataset_name=dataset,
        #     mode='Test',
        #     save_path=f'{dataset}_{metric}_test.pdf'
        # )


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import os
methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
        'LexicalSimilarity_rougeL': 'LSRL'
}

methrics_dict ={
  'MSP':'MSP', 'PPL' :'PPL', 'MTE' :'MTE',  'MCSE':'MCSE', 'MCNSE':'MCNSE', 'LSRL': 'LSRL'
}

DATASETS=['xsum']
models =['llama','gemma']
for model in models:
    for dataset in DATASETS:
        train_ue_values, train_metric_values, train_gen_lengths = extract_and_prepare_data_hello(dataset, methods_dict, all_metrics, model=model, suff='full_enriched')

        for metric, metric_short  in methods_dict.items():
            os.makedirs(f'{model}/{metric}', exist_ok=True)
            plot_metric_vs_length(
                gen_lengths=np.array(train_gen_lengths),
                metric_values=np.array(train_ue_values[metric_short]),
                metric_name=metric_short,
                dataset_name=dataset,
                mode='Train',
                save_path=f'{model}/{metric}/{dataset}_{metric}_{model}_train.pdf',
                # model = model
            )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from collections import defaultdict
from utils import extract_and_prepare_data, detrend_ue
from pathlib import Path
import pathlib
import argparse

normalize = True

methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
    'LexicalSimilarity_rougeL': 'LSRL',
}

MODELS = {
    'llama': 'Llama 3.1 8B',
    'gemma': 'Gemma 2 9B',
    'eurollm': 'EuroLLM 9B',
}

DATASETS = {
    'metricx-metricx-24-hybrid-xxl-v2p6': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'XComet-XCOMET-XXL': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'Comet-wmt22-comet-da': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'bleu_proper': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
}

METRICS = {
    'metricx-metricx-24-hybrid-xxl-v2p6': 'MetricX XXL',
    'XComet-XCOMET-XXL': 'XComet XXL',
    'Comet-wmt22-comet-da': 'Comet WMT22',
    'bleu_proper': 'BLEU',
}

pathlib.Path('tables').mkdir(parents=True, exist_ok=True)
pathlib.Path('charts').mkdir(parents=True, exist_ok=True)

def get_header(caption):
    return (
        "\\begin{table*}\n"
        "\\footnotesize\n"
        f"\caption{{{caption}}}\n"
        "\\begin{tabular}{lcccccccc}\n"
        "&\multicolumn{4}{c}{\\textbf{WMT14}}&\multicolumn{4}{c}{\\textbf{WMT19}}\\\\\n"
        "\cmidrule(lr){2-5}\n"
        "\cmidrule(lr){6-9}\n"
    )

def footer():
    return (
        "\midrule\n"
        "\end{tabular}\n"
        "\end{table*}\n"
    )

def colname(dataset):
    if '_' in dataset:
        dataset = dataset.split('_')[1]

    return dataset[:2].capitalize() + '-' + dataset[2:].capitalize()


def main(type_):
    for metric, metric_name in METRICS.items():
        datasets = DATASETS[metric]
        ue_methods = list(methods_dict.values())

        if type_ == 'all':
            caption = f"PRR improvement (detrended $-$ raw) across all methods. Metric is {metric_name}."
            header = get_header(caption)
            header += "&" + "&".join([colname(dataset) for dataset in datasets]) + "\\\\\n"
            latex = header

            for model_type in ['base']:
                for model, model_name in MODELS.items():
                    model_title = model_name if model_type == 'base' else f"{model_name} Instruct"
                    latex += "\midrule\n"
                    latex += "& \\multicolumn{8}{c}{" + model_title + "}\\\\\n"
                    latex += "\midrule\n"

                    ue_scores, _, _ = detrend_ue(
                        datasets, model, model_type,
                        [metric], ue_methods, methods_dict, return_unprocessed=True
                    )

                    for method_full, method_short in methods_dict.items():
                        row = [method_short]
                        for i, dataset in enumerate(datasets):
                            raw = ue_scores[f"{method_short}_raw"][i]
                            detr = ue_scores[f"{method_short}_detr"][i]
                            delta = detr - raw
                            formatted = f"{delta:+.2f}"
                            if delta > 0:
                                formatted = f"\\textbf{{{formatted}}}"
                            row.append(formatted)
                        latex += " & ".join(row) + "\\\\\n"

            latex += footer()
            name = f'tables/{metric}_all_ue_deltas.tex'


        with open(name, 'w') as f:
            f.write(latex)

if __name__ == "__main__":
    type_='all'
    main(type_)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from collections import defaultdict
from utils import extract_and_prepare_data, detrend_ue, detrend_ue_w_quality
from pathlib import Path
import pathlib
import argparse

normalize = True

methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
    'LexicalSimilarity_rougeL': 'LSRL',
}

MODELS = {
    'llama': 'Llama 3.1 8B',
    'gemma': 'Gemma 2 9B',
    'eurollm': 'EuroLLM 9B',
}

DATASETS = {
    'metricx-metricx-24-hybrid-xxl-v2p6': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'XComet-XCOMET-XXL': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'Comet-wmt22-comet-da': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'bleu_proper': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
}

METRICS = {
    'metricx-metricx-24-hybrid-xxl-v2p6': 'MetricX XXL',
    'XComet-XCOMET-XXL': 'XComet XXL',
    'Comet-wmt22-comet-da': 'Comet WMT22',
    # 'bleu_proper': 'BLEU',
}

pathlib.Path('tables').mkdir(parents=True, exist_ok=True)
pathlib.Path('charts').mkdir(parents=True, exist_ok=True)

def get_header(caption):
    return (
        "\\begin{table*}\n"
        "\\footnotesize\n"
        "\\centering\n"
        "\\toprule\n"
        f"\caption{{{caption}}}\n"
        "\\begin{tabular}{lcccccccc}\n"
        "&\multicolumn{4}{c}{\\textbf{WMT14}}&\multicolumn{4}{c}{\\textbf{WMT19}}\\\\\n"
        "\cmidrule(lr){2-5}\n"
        "\cmidrule(lr){6-9}\n"
    )

def footer():
    return (
        "\midrule\n"
        "\end{tabular}\n"
        "\end{table*}\n"
    )

def colname(dataset):
    if '_' in dataset:
        dataset = dataset.split('_')[1]

    return dataset[:2].capitalize() + '-' + dataset[2:].capitalize()


def main(type_):
    for metric, metric_name in METRICS.items():
        datasets = DATASETS[metric]
        ue_methods = list(methods_dict.values())

        caption = f"Detailed PRR scores (raw and detrended) for all methods. Metric: {metric_name}."
        header = get_header(caption)
        header += "&" + "&".join([colname(dataset).replace('_', '\\_') for dataset in datasets]) + "\\\\\n"
        latex = header

        for model_type in ['base']:
            for model, model_name in MODELS.items():
                model_title = model_name if model_type == 'base' else f"{model_name} Instruct"
                latex += "\midrule\n"
                latex += "& \\multicolumn{8}{c}{" + model_title + "}\\\\\n"
                latex += "\midrule\n"

                ue_scores, _, _ = detrend_ue_w_quality(datasets, model, model_type, [metric], ue_methods, methods_dict, return_unprocessed=True)

                # Find best and second-best scores per dataset (column-wise)
                best_map = {}
                second_map = {}
                for col_idx in range(len(datasets)):
                    scores = []
                    for method_short in methods_dict.values():
                        raw = ue_scores[f"{method_short}_raw"][col_idx]
                        detr = ue_scores[f"{method_short}_detr"][col_idx]
                        scores.extend([raw, detr])
                    sorted_scores = sorted(scores, reverse=True)
                    best_map[col_idx] = sorted_scores[0]
                    second_map[col_idx] = sorted_scores[1]

                for method_full, method_short in methods_dict.items():
                    raw_scores = ue_scores[f"{method_short}_raw"]
                    detr_scores = ue_scores[f"{method_short}_detr"]

                    raw_row = [f"Raw {method_short}"]
                    detr_row = [f"Detr {method_short}"]

                    for col_idx, (raw, detr) in enumerate(zip(raw_scores, detr_scores)):
                        # Format raw
                        if np.isclose(raw, best_map[col_idx]):
                            raw_row.append(f"\\textbf{{{raw:.2f}}}")
                        elif np.isclose(raw, second_map[col_idx]):
                            raw_row.append(f"\\underline{{{raw:.2f}}}")
                        else:
                            raw_row.append(f"{raw:.2f}")

                        # Format detr
                        if np.isclose(detr, best_map[col_idx]):
                            detr_row.append(f"\\textbf{{{detr:.2f}}}")
                        elif np.isclose(detr, second_map[col_idx]):
                            detr_row.append(f"\\underline{{{detr:.2f}}}")
                        else:
                            detr_row.append(f"{detr:.2f}")

                    latex += " & ".join(raw_row) + " \\\\\n"
                    latex += " & ".join(detr_row) + " \\\\\n"
                    latex += "\\midrule\n"

        latex += footer()
        name = f'tables/{metric}_detailed_results_per_dataset_ranking_w_quality.tex'
        with open(name, 'w') as f:
            f.write(latex)


if __name__ == "__main__":
    type_='all'
    main(type_)

In [6]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from collections import defaultdict
from utils import extract_and_prepare_data, detrend_ue
from pathlib import Path
import pathlib
import argparse

normalize = True

methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
    'LexicalSimilarity_rougeL': 'LSRL',
}

MODELS = {
    'llama': 'Llama 3.1 8B',
    'gemma': 'Gemma 2 9B',
    'eurollm': 'EuroLLM 9B',
}

DATASETS = {
    'metricx-metricx-24-hybrid-xxl-v2p6': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'XComet-XCOMET-XXL': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'Comet-wmt22-comet-da': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'bleu_proper': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
}

METRICS = {
    'metricx-metricx-24-hybrid-xxl-v2p6': 'MetricX XXL',
    'XComet-XCOMET-XXL': 'XComet XXL',
    'Comet-wmt22-comet-da': 'Comet WMT22',
    'bleu_proper': 'BLEU',
}

pathlib.Path('tables').mkdir(parents=True, exist_ok=True)
pathlib.Path('charts').mkdir(parents=True, exist_ok=True)

def get_header(caption):
    return (
        "\\begin{table*}\n"
        "\\footnotesize\n"
        f"\caption{{{caption}}}\n"
        "\\begin{tabular}{lcccccccc}\n"
        "&\multicolumn{4}{c}{\\textbf{WMT14}}&\multicolumn{4}{c}{\\textbf{WMT19}}\\\\\n"
        "\cmidrule(lr){2-5}\n"
        "\cmidrule(lr){6-9}\n"
    )

def footer():
    return (
        "\midrule\n"
        "\end{tabular}\n"
        "\end{table*}\n"
    )

def colname(dataset):
    if '_' in dataset:
        dataset = dataset.split('_')[1]

    return dataset[:2].capitalize() + '-' + dataset[2:].capitalize()
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from collections import defaultdict
from utils import extract_and_prepare_data, detrend_ue
from pathlib import Path
import pathlib
import argparse

normalize = True

methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
    'LexicalSimilarity_rougeL': 'LSRL',
}

MODELS = {
    'llama': 'Llama 3.1 8B',
    'gemma': 'Gemma 2 9B',
    'eurollm': 'EuroLLM 9B',
}

DATASETS = {
    'metricx-metricx-24-hybrid-xxl-v2p6': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'XComet-XCOMET-XXL': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'Comet-wmt22-comet-da': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'bleu_proper': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
}

METRICS = {
    'metricx-metricx-24-hybrid-xxl-v2p6': 'MetricX XXL',
    'XComet-XCOMET-XXL': 'XComet XXL',
    'Comet-wmt22-comet-da': 'Comet WMT22',
    'bleu_proper': 'BLEU',
}

pathlib.Path('tables').mkdir(parents=True, exist_ok=True)
pathlib.Path('charts').mkdir(parents=True, exist_ok=True)

def get_header(caption):
    return (
        "\\begin{table*}\n"
        "\\footnotesize\n"
        f"\caption{{{caption}}}\n"
        "\\begin{tabular}{lcccccccc}\n"
        "&\multicolumn{4}{c}{\\textbf{WMT14}}&\multicolumn{4}{c}{\\textbf{WMT19}}\\\\\n"
        "\cmidrule(lr){2-5}\n"
        "\cmidrule(lr){6-9}\n"
    )

def footer():
    return (
        "\midrule\n"
        "\end{tabular}\n"
        "\end{table*}\n"
    )

def colname(dataset):
    if '_' in dataset:
        dataset = dataset.split('_')[1]

    return dataset[:2].capitalize() + '-' + dataset[2:].capitalize()

def main(type_):
    for metric, metric_name in METRICS.items():
        datasets = DATASETS[metric]
        ue_methods = list(methods_dict.values())

        if type_ == 'all':
            model_types = ['base']
            for model_type in model_types:
                caption = f"Average PRR improvement and rank gain from detrending across models. Metric: {metric_name}."
                latex = "\\begin{table*}[ht]\n\\centering\n\\small\n"
                latex += f"\\caption{{{caption}}}\n"
                latex += "\\begin{tabular}{{l" + "cc" * len(MODELS) + "}}\n"
                latex += "\\toprule\n"
                latex += "Method"

                for model in MODELS.values():
                    latex += f" & \\multicolumn{{2}}{{c}}{{{model}}}"

                latex += " \\\\\n"
                latex += " & " + " & ".join(["$\\Delta$PRR", "Rank Gain"] * len(MODELS)) + " \\\\\n"
                latex += "\\midrule\n"

                for method in ue_methods:
                    row = f"{method}"
                    for model in MODELS.keys():
                        ue_scores, _, _ = detrend_ue(
                            datasets, model, model_type,
                            [metric], ue_methods, methods_dict, return_unprocessed=True
                        )

                        deltas = []
                        rank_gains = []

                        for i in range(len(datasets)):
                            raw = ue_scores[f"{method}_raw"][i]
                            detr = ue_scores[f"{method}_detr"][i]
                            delta = detr - raw
                            deltas.append(delta)

                            # Rank gain calculation
                            combined = []
                            for m in ue_methods:
                                combined.append(ue_scores[f"{m}_raw"][i])
                                combined.append(ue_scores[f"{m}_detr"][i])
                            ranks = pd.Series(combined).rank(ascending=False, method='min').values

                            idx_raw = 2 * ue_methods.index(method)
                            idx_detr = idx_raw + 1
                            gain = ranks[idx_raw] - ranks[idx_detr]
                            rank_gains.append(gain)

                        delta_mean = np.mean(deltas)
                        delta_std = np.std(deltas)
                        gain_mean = np.mean(rank_gains)
                        gain_std = np.std(rank_gains)

                        delta_fmt = f"\\textbf{{{delta_mean:+.2f}}} $\\pm$ {delta_std:.2f}" if delta_mean > 0 else f"{delta_mean:+.2f} $\\pm$ {delta_std:.2f}"
                        gain_fmt = f"{gain_mean:.2f} $\\pm$ {gain_std:.2f}"

                        row += f" & {delta_fmt} & {gain_fmt}"

                    latex += row + " \\\\\n"

                latex += "\\bottomrule\n\\end{tabular}\n\\end{table*}\n"

                with open(f"tables/{metric}_combined_models_avg_deltas_and_ranks.tex", 'w') as f:
                    f.write(latex)
if __name__ == "__main__":
    type_='all'
    main(type_)



/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f2f008e1f60>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f2eaea69330>]
llama wmt14_csen Below q ids: 1771


/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


...

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f2eaea681c0>]
eurollm wmt14_deen Below q ids: 1748


/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f2eaeaa82e0>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


KeyboardInterrupt: 

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from collections import defaultdict
from utils import extract_and_prepare_data, detrend_ue
from pathlib import Path
import pathlib
import argparse

normalize = True

methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
    'LexicalSimilarity_rougeL': 'LSRL',
}

MODELS = {
    'llama': 'Llama 3.1 8B',
    'gemma': 'Gemma 2 9B',
    'eurollm': 'EuroLLM 9B',
}

DATASETS = {
    'metricx-metricx-24-hybrid-xxl-v2p6': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'XComet-XCOMET-XXL': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'Comet-wmt22-comet-da': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'bleu_proper': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
}

METRICS = {
    'metricx-metricx-24-hybrid-xxl-v2p6': 'MetricX XXL',
    'XComet-XCOMET-XXL': 'XComet XXL',
    'Comet-wmt22-comet-da': 'Comet WMT22',
    'bleu_proper': 'BLEU',
}

pathlib.Path('tables').mkdir(parents=True, exist_ok=True)
pathlib.Path('charts').mkdir(parents=True, exist_ok=True)

def get_header(caption):
    return (
        "\\begin{table*}\n"
        "\\footnotesize\n"
        f"\caption{{{caption}}}\n"
        "\\begin{tabular}{lcccccccc}\n"
        "&\multicolumn{4}{c}{\\textbf{WMT14}}&\multicolumn{4}{c}{\\textbf{WMT19}}\\\\\n"
        "\cmidrule(lr){2-5}\n"
        "\cmidrule(lr){6-9}\n"
    )

def footer():
    return (
        "\midrule\n"
        "\end{tabular}\n"
        "\end{table*}\n"
    )

def colname(dataset):
    if '_' in dataset:
        dataset = dataset.split('_')[1]

    return dataset[:2].capitalize() + '-' + dataset[2:].capitalize()

def format_score_pair(raw, detr, threshold=0.03):
    arrow = "$\\uparrow$" if detr - raw > threshold else ""
    return f"{raw:.2f}", f"{detr:.2f}{arrow}"

def main(type_):
    for metric, metric_name in METRICS.items():
        datasets = DATASETS[metric]
        ue_methods = list(methods_dict.values())

        caption = f"Best raw and detrended PRR scores per dataset and model. Metric: {metric_name}."
        latex = "\\begin{table*}[ht]\n\\centering\n\\small\n"
        latex += f"\\caption{{{caption}}}\n"
        latex += "\\begin{tabular}{l" + "cc" * len(datasets) + "}\n"
        latex += "\\toprule\n"
        
        # First header row: dataset groups
        group_line = "Model"
        for _ in datasets:
            group_line += " & \\multicolumn{2}{c}{}"
        latex += group_line + " \\\\\n"

        dataset_line = " "
        for d in datasets:
            name = colname(d)
            dataset_line += f" & \\multicolumn{{2}}{{c}}{{{name}}}"
        latex += dataset_line + " \\\\\n"

        metric_line = " "
        for _ in datasets:
            metric_line += " & Raw & Detr"
        latex += metric_line + " \\\\\n\\midrule\n"

        for model_key, model_name in MODELS.items():
            latex += f"\\midrule\n\\multicolumn{{{1 + 2*len(datasets)}}}{{c}}{{\\textbf{{{model_name}}}}} \\\\\n\\midrule\n"

            ue_scores, _, _ = detrend_ue(
                datasets, model_key, "base", [metric], ue_methods, methods_dict, return_unprocessed=True
            )

            row = [" "]
            for i in range(len(datasets)):
                best_raw = -float("inf")
                best_detr = -float("inf")

                for method in ue_methods:
                    raw = ue_scores[f"{method}_raw"][i]
                    detr = ue_scores[f"{method}_detr"][i]
                    best_raw = max(best_raw, raw)
                    best_detr = max(best_detr, detr)

                raw_fmt, detr_fmt = format_score_pair(best_raw, best_detr)
                row.extend([raw_fmt, detr_fmt])

            latex += " & ".join(row) + " \\\\\n"

        latex += "\\bottomrule\n\\end{tabular}\n\\end{table*}\n"

        with open(f"tables/{metric}_combined_model_results.tex", "w") as f:
            f.write(latex)

if __name__ == "__main__":
    type_='all'
    main(type_)

/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f5f453216f0>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f5f9517d4b0>]
llama wmt14_csen Below q ids: 1771


/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


...

Error during conversion: ChunkedEncodingError(ProtocolError('Response ended prematurely'))
/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f61c3dea9e0>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f5f98878d60>]
eurollm wmt19_ruen Below q ids: 1762


Error during conversion: ChunkedEncodingError(ProtocolError('Response ended prematurely'))


In [37]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from collections import defaultdict
from utils import extract_and_prepare_data, detrend_ue
from pathlib import Path
import pathlib
import argparse

normalize = True

methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
    'LexicalSimilarity_rougeL': 'LSRL',
}

MODELS = {
    'llama': 'Llama 3.1 8B',
    'gemma': 'Gemma 2 9B',
    'eurollm': 'EuroLLM 9B',
}

DATASETS = {
    'metricx-metricx-24-hybrid-xxl-v2p6': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'XComet-XCOMET-XXL': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'Comet-wmt22-comet-da': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'bleu_proper': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
}

METRICS = {
    'metricx-metricx-24-hybrid-xxl-v2p6': 'MetricX XXL',
    'XComet-XCOMET-XXL': 'XComet XXL',
    'Comet-wmt22-comet-da': 'Comet WMT22',
    # 'bleu_proper': 'BLEU',
}

pathlib.Path('tables').mkdir(parents=True, exist_ok=True)
pathlib.Path('charts').mkdir(parents=True, exist_ok=True)

def get_header(caption):
    return (
        "\\begin{table*}\n"
        "\\footnotesize\n"
        f"\caption{{{caption}}}\n"
        "\\begin{tabular}{lcccccccc}\n"
        "&\multicolumn{4}{c}{\\textbf{WMT14}}&\multicolumn{4}{c}{\\textbf{WMT19}}\\\\\n"
        "\cmidrule(lr){2-5}\n"
        "\cmidrule(lr){6-9}\n"
    )

def footer():
    return (
        "\midrule\n"
        "\end{tabular}\n"
        "\end{table*}\n"
    )

def colname(dataset):
    if '_' in dataset:
        dataset = dataset.split('_')[1]

    return dataset[:2].capitalize() + '-' + dataset[2:].capitalize()
def format_score_pair(raw, detr, threshold=0.00):
    arrow = "$\\uparrow$" if detr - raw > threshold else ""
    return f"{raw:.2f}", f"{detr:.2f}{arrow}"

def main(type_):
    caption = "Best raw and detrended PRR scores across all metrics and models."
    latex = "\\begin{table*}[ht]\n\\centering\n\\small\n"
    latex += f"\\caption{{{caption}}}\n"
    total_cols = max(len(v) for v in DATASETS.values())
    col_spec = "l" + "cc" * total_cols
    latex += f"\\begin{{tabular}}{{{col_spec}}}\n\\toprule\n"

    # Dataset header (generic, since all blocks will have same layout)
    dataset_line = "Model"
    subheader_line = " "
    example_metric = next(iter(METRICS))
    for d in DATASETS[example_metric]:
        name = colname(d)
        dataset_line += f" & \\multicolumn{{2}}{{c}}{{{name}}}"
        subheader_line += " & Raw & Detr"
    latex += dataset_line + " \\\\\n"
    latex += subheader_line + " \\\\\n\\midrule\n"

    for metric, metric_name in METRICS.items():
        datasets = DATASETS[metric]
        ue_methods = list(methods_dict.values())

        # Section header for metric
        n_cols = 1 + 2 * len(datasets)
        latex += f"\\multicolumn{{{n_cols}}}{{c}}{{\\textbf{{{metric_name}}}}} \\\\\n"

        for model_key, model_name in MODELS.items():
            ue_scores, _, _ = detrend_ue(
                datasets, model_key, "base", [metric], ue_methods, methods_dict, return_unprocessed=True
            )

            row = [model_name]
            for i in range(len(datasets)):
                best_raw = -float("inf")
                best_detr = -float("inf")

                for method in ue_methods:
                    raw = ue_scores[f"{method}_raw"][i]
                    detr = ue_scores[f"{method}_detr"][i]
                    best_raw = max(best_raw, raw)
                    best_detr = max(best_detr, detr)

                raw_fmt, detr_fmt = format_score_pair(best_raw, best_detr)
                row.extend([raw_fmt, detr_fmt])

            latex += " & ".join(row) + " \\\\\n"

        latex += "\\midrule\n"

    latex += "\\bottomrule\n\\end{tabular}\n\\end{table*}\n"

    with open("tables/final_rowwise_metric_blocks.tex", "w") as f:
        f.write(latex)



if __name__ == "__main__":
    type_='all'
    main(type_)

/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f5f98556bc0>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f5f44c5d1e0>]
llama wmt14_csen Below q ids: 1771


/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


...

/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f5f44b42fb0>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f5f983026e0>]
eurollm wmt19_ruen Below q ids: 1762


Error during conversion: ChunkedEncodingError(ProtocolError('Response ended prematurely'))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from collections import defaultdict
from utils import extract_and_prepare_data, detrend_ue, detrend_ue_w_quality
from pathlib import Path
import pathlib
import argparse
from sklearn.linear_model import LinearRegression

normalize = True

methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
    'LexicalSimilarity_rougeL': 'LSRL',
}

MODELS = {
    'llama': 'Llama 3.1 8B',
    'gemma': 'Gemma 2 9B',
    # 'eurollm': 'EuroLLM 9B',
}

DATASETS = {
    'metricx-metricx-24-hybrid-xxl-v2p6': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'XComet-XCOMET-XXL': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'Comet-wmt22-comet-da': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'bleu_proper': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'AlignScoreInputOutput' : ['xsum'],
        'Accuracy' : ['gsm8k'],
            'Rouge_rougeL' : ['xsum'],


}

METRICS = {
                'Rouge_rougeL' : 'Rouge L',
    'AlignScoreInputOutput': 'Align Score',
    # 'XComet-XCOMET-XXL': 'XComet XXL',
    'Accuracy':'Acc'
    # 'Comet-wmt22-comet-da': 'Comet WMT22',
    # 'bleu_proper': 'BLEU',
}

pathlib.Path('tables').mkdir(parents=True, exist_ok=True)
pathlib.Path('charts').mkdir(parents=True, exist_ok=True)

def get_header(caption):
    return (
        "\\begin{table*}\n"
        "\\footnotesize\n"
        "\\centering\n"
        "\\toprule\n"
        f"\caption{{{caption}}}\n"
        "\\begin{tabular}{lcccccccc}\n"
        "&\multicolumn{4}{c}{\\textbf{WMT14}}&\multicolumn{4}{c}{\\textbf{WMT19}}\\\\\n"
        "\cmidrule(lr){2-5}\n"
        "\cmidrule(lr){6-9}\n"
    )

def footer():
    return (
        "\midrule\n"
        "\end{tabular}\n"
        "\end{table*}\n"
    )

def colname(dataset):
    if '_' in dataset:
        dataset = dataset.split('_')[1]

    return dataset[:2].capitalize() + '-' + dataset[2:].capitalize()

def plot_raw_vs_detrended(lengths, raw_scores, detr_scores, method_name, save_path=None):
    lengths = np.array(lengths).reshape(-1, 1)
    raw_scores = np.array(raw_scores)
    detr_scores = np.array(detr_scores)

    # Fit linear regression models
    reg_raw = LinearRegression().fit(lengths, raw_scores)
    reg_detr = LinearRegression().fit(lengths, detr_scores)

    # Predict
    raw_fit = reg_raw.predict(lengths)
    detr_fit = reg_detr.predict(lengths)

    plt.figure(figsize=(8, 6))
    plt.scatter(lengths, raw_scores, label='Raw Scores', alpha=0.6, color='blue')
    plt.plot(lengths, raw_fit, color='blue', linestyle='--', label=f'Raw Fit (slope={reg_raw.coef_[0]:.3f})')

    plt.scatter(lengths, detr_scores, label='Detrended Scores', alpha=0.6, color='green')
    plt.plot(lengths, detr_fit, color='green', linestyle='--', label=f'Detrended Fit (slope={reg_detr.coef_[0]:.3f})')

    plt.title(f'Length Effect for {method_name}')
    plt.xlabel('Sequence Length')
    plt.ylabel('Uncertainty Score')
    plt.legend()
    plt.grid(True)

    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    else:
        plt.show()


def main(type_):
    for metric, metric_name in METRICS.items():
        datasets = DATASETS[metric]
        ue_methods = list(methods_dict.values())

        caption = f"Detailed PRR scores (raw and detrended) for all methods. Metric: {metric_name}."
        header = get_header(caption)
        header += "&" + "&".join([colname(dataset).replace('_', '\\_') for dataset in datasets]) + "\\\\\n"
        latex = header

        for model_type in ['base']:
            for model, model_name in MODELS.items():
                model_title = model_name if model_type == 'base' else f"{model_name} Instruct"
                latex += "\midrule\n"
                latex += "& \\multicolumn{8}{c}{" + model_title + "}\\\\\n"
                latex += "\midrule\n"

                ue_scores, _, _,ue_scores_full,length_test = detrend_ue_w_quality(datasets, model, model_type, [metric], ue_methods, methods_dict, return_unprocessed=True)

                # Find best and second-best scores per dataset (column-wise)
                best_map = {}
                second_map = {}
                for col_idx in range(len(datasets)):
                    scores = []
                    for method_short in methods_dict.values():
                        raw = ue_scores[f"{method_short}_raw"][col_idx]
                        detr = ue_scores[f"{method_short}_detr"][col_idx]
                        scores.extend([raw, detr])
                        print(method_short, ':', raw, ' -> ', detr)
                    sorted_scores = sorted(scores, reverse=True)
                    best_map[col_idx] = sorted_scores[0]
                    second_map[col_idx] = sorted_scores[1]

                for method_full, method_short in methods_dict.items():
                    raw_scores = ue_scores[f"{method_short}_raw"]
                    detr_scores = ue_scores[f"{method_short}_detr"]
                    # plot_raw_vs_detrended(length_test, ue_scores_full[f"{method_short}_raw"], ue_scores_full[f"{method_short}_detr"], method_name=method_short)
                    # print(raw_scores, detr_scores, length_test)
                    
                    raw_row = [f"Raw {method_short}"]
                    detr_row = [f"Detr {method_short}"]

                    for col_idx, (raw, detr) in enumerate(zip(raw_scores, detr_scores)):
                        # Format raw
                        if np.isclose(raw, best_map[col_idx]):
                            raw_row.append(f"\\textbf{{{raw:.2f}}}")
                        elif np.isclose(raw, second_map[col_idx]):
                            raw_row.append(f"\\underline{{{raw:.2f}}}")
                        else:
                            raw_row.append(f"{raw:.2f}")

                        # Format detr
                        if np.isclose(detr, best_map[col_idx]):
                            detr_row.append(f"\\textbf{{{detr:.2f}}}")
                        elif np.isclose(detr, second_map[col_idx]):
                            detr_row.append(f"\\underline{{{detr:.2f}}}")
                        else:
                            detr_row.append(f"{detr:.2f}")

                    latex += " & ".join(raw_row) + " \\\\\n"
                    latex += " & ".join(detr_row) + " \\\\\n"
                    latex += "\\midrule\n"

        latex += footer()
        name = f'tables/{metric}_detailed_results_per_dataset_ranking_w_quality.tex'
        with open(name, 'w') as f:
            f.write(latex)


if __name__ == "__main__":
    type_='all'
    main(type_)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from collections import defaultdict
from utils import extract_and_prepare_data, detrend_ue, detrend_ue_w_quality, detrend_ue_w_quality_only
from pathlib import Path
import pathlib
import argparse
from sklearn.linear_model import LinearRegression

normalize = True

methods_dict = {
    'MaximumSequenceProbability': 'MSP',
    'Perplexity': 'PPL',
    'MeanTokenEntropy': 'MTE',
    'MonteCarloSequenceEntropy': 'MCSE',
    'MonteCarloNormalizedSequenceEntropy': 'MCNSE',
    'LexicalSimilarity_rougeL': 'LSRL',
}

MODELS = {
    'llama': 'Llama 3.1 8B',
    'gemma': 'Gemma 2 9B',
    # 'eurollm': 'EuroLLM 9B',
}

DATASETS = {
    'metricx-metricx-24-hybrid-xxl-v2p6': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'XComet-XCOMET-XXL': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'Comet-wmt22-comet-da': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'bleu_proper': [
        'wmt14_csen',
        'wmt14_deen',
        'wmt14_ruen',
        'wmt14_fren',
        'wmt19_deen',
        'wmt19_fien',
        'wmt19_lten',
        'wmt19_ruen',
    ],
    'AlignScoreInputOutput' : ['xsum'],
        'Accuracy' : ['gsm8k'],
            # 'Rouge_rougeL' : ['xsum'],


}

METRICS = {
                # 'Rouge_rougeL' : 'Rouge L',
    'AlignScoreInputOutput': 'Align Score',
    # 'XComet-XCOMET-XXL': 'XComet XXL',
    'Accuracy':'Acc',
    # 'metricx-metricx-24-hybrid-xxl-v2p6': 'MetricX-XXL',
    # 'Comet-wmt22-comet-da' :'Comet WMT22',
    # 'bleu_proper': 'BLEU',
}

pathlib.Path('tables').mkdir(parents=True, exist_ok=True)
pathlib.Path('charts').mkdir(parents=True, exist_ok=True)

def get_header(caption):
    return (
        "\\begin{table*}\n"
        "\\footnotesize\n"
        "\\centering\n"
        "\\toprule\n"
        f"\caption{{{caption}}}\n"
        "\\begin{tabular}{lcccccccc}\n"
        "&\multicolumn{4}{c}{\\textbf{WMT14}}&\multicolumn{4}{c}{\\textbf{WMT19}}\\\\\n"
        "\cmidrule(lr){2-5}\n"
        "\cmidrule(lr){6-9}\n"
    )

def footer():
    return (
        "\midrule\n"
        "\end{tabular}\n"
        "\end{table*}\n"
    )

def colname(dataset):
    if '_' in dataset:
        dataset = dataset.split('_')[1]

    return dataset[:2].capitalize() + '-' + dataset[2:].capitalize()

def plot_raw_vs_detrended(lengths, raw_scores, detr_scores, method_name, save_path=None):
    lengths = np.array(lengths).reshape(-1, 1)
    raw_scores = np.array(raw_scores)
    detr_scores = np.array(detr_scores)

    # Fit linear regression models
    reg_raw = LinearRegression().fit(lengths, raw_scores)
    reg_detr = LinearRegression().fit(lengths, detr_scores)

    # Predict
    raw_fit = reg_raw.predict(lengths)
    detr_fit = reg_detr.predict(lengths)

    plt.figure(figsize=(8, 6))
    plt.scatter(lengths, raw_scores, label='Raw Scores', alpha=0.6, color='blue')
    plt.plot(lengths, raw_fit, color='blue', linestyle='--', label=f'Raw Fit (slope={reg_raw.coef_[0]:.3f})')

    plt.scatter(lengths, detr_scores, label='Detrended Scores', alpha=0.6, color='green')
    plt.plot(lengths, detr_fit, color='green', linestyle='--', label=f'Detrended Fit (slope={reg_detr.coef_[0]:.3f})')

    plt.title(f'Length Effect for {method_name}')
    plt.xlabel('Sequence Length')
    plt.ylabel('Uncertainty Score')
    plt.legend()
    plt.grid(True)

    if save_path:
        plt.savefig(save_path, bbox_inches='tight')
    else:
        plt.show()


def main(type_):
    results = []
    for metric, metric_name in METRICS.items():
        datasets = DATASETS[metric]
        ue_methods = list(methods_dict.values())

        caption = f"Detailed PRR scores (raw and detrended) for all methods. Metric: {metric_name}."
        header = get_header(caption)
        header += "&" + "&".join([colname(dataset).replace('_', '\\_') for dataset in datasets]) + "\\\\\n"
        latex = header

        for model_type in ['base']:
            for model, model_name in MODELS.items():
                model_title = model_name if model_type == 'base' else f"{model_name} Instruct"
                latex += "\midrule\n"
                latex += "& \\multicolumn{8}{c}{" + model_title + "}\\\\\n"
                latex += "\midrule\n"

                ue_scores, _, _, _,_ = detrend_ue_w_quality(datasets, model, model_type, [metric], ue_methods, methods_dict, return_unprocessed=True, quality_fit_sample_size=100)

                # Find best and second-best scores per dataset (column-wise)
                best_map = {}
                second_map = {}
                for col_idx in range(len(datasets)):
                    scores = []
                    for method_short in methods_dict.values():
                        raw = ue_scores[f"{method_short}_raw"][col_idx]
                        detr = ue_scores[f"{method_short}_detr"][col_idx]
                        scores.extend([raw, detr])
                        print( method_short, metric,datasets[col_idx], ':', raw, ' -> ', detr)
                        results.append({
                            "Metric": metric_name,
                            "Dataset": datasets[col_idx],
                            "Model": model_title,
                            "Method": method_short,
                            "ScoreType": "Raw",
                            "Score": raw
                        })

                        results.append({
                            "Metric": metric_name,
                            "Dataset": datasets[col_idx],
                            "Model": model_title,
                            "Method": method_short,
                            "ScoreType": "Detrended",
                            "Score": detr
                        })
        df = pd.DataFrame(results)
        csv_path = "ue_scores_results_quality_100.csv"
        df.to_csv(csv_path, index=False)
        print(f"Results saved to {csv_path}")

                    # sorted_scores = sorted(scores, reverse=True)
                    # best_map[col_idx] = sorted_scores[0]
                    # second_map[col_idx] = sorted_scores[1]

    


if __name__ == "__main__":
    type_='all'
    main(type_)

/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/maiya.goloburda/.conda/envs/detrend/lib/python3.10/site-packages/torch/cuda/__init__.py:734: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a Bert

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f736023f460>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f736240b6d0>]
llama xsum Below q ids: 1620


Error during conversion: ChunkedEncodingError(ProtocolError('Response ended prematurely'))


...

Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f730df729b0>]


Some weights of the model checkpoint at microsoft/deberta-large-mnli were not used when initializing DebertaForSequenceClassification: ['config']
- This IS expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DebertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Stat calculators: [<lm_polygraph.stat_calculators.greedy_probs.GreedyProbsCalculator object at 0x7f7360250fa0>]
gemma gsm8k Below q ids: 1798


Error during conversion: ChunkedEncodingError(ProtocolError('Response ended prematurely'))


MSP Accuracy gsm8k : 0.3028441948367712  ->  0.21277462186540982
PPL Accuracy gsm8k : 0.24825509077116434  ->  0.19999070144466094
MTE Accuracy gsm8k : 0.29151407538513324  ->  0.24138388542708517
MCSE Accuracy gsm8k : 0.3932644724525617  ->  0.29798016538205324
MCNSE Accuracy gsm8k : 0.3560989881486637  ->  0.2534256831072676
LSRL Accuracy gsm8k : 0.39424192349781356  ->  0.28523145610525635
Results saved to ue_scores_results_quality_100.csv


In [2]:
import pandas as pd 

df = pd.read_csv('ue_scores_results_quality_100.csv')

df.head()

,Metric,Dataset,Model,Method,ScoreType,Score
0,Align Score,xsum,Llama 3.1 8B,MSP,Raw,0.327603
1,Align Score,xsum,Llama 3.1 8B,MSP,Detrended,0.345266
2,Align Score,xsum,Llama 3.1 8B,PPL,Raw,0.369137
3,Align Score,xsum,Llama 3.1 8B,PPL,Detrended,0.363297
4,Align Score,xsum,Llama 3.1 8B,MTE,Raw,0.356930


In [ ]:
import pandas as pd

# Load the CSV
df = pd.read_csv('ue_scores_results_quality_100.csv')

# Extract only the detrended columns
detr_cols = [col for col in df.columns if col.endswith('_detr')]

# Get model and dataset identifiers (assuming they are in the index or columns)
models = df['Model'].unique() if 'model' in df.columns else ['Gemma', 'LLaMA']
datasets = df['Dataset'].unique() if 'dataset' in df.columns else ['GSM8k', 'XSUM']

# Prepare results container
latex_row = []

# Loop through each dataset-model pair
for dataset in datasets:
    for model in models:
        subset = df[(df['Dataset'] == dataset) & (df['Model'] == model)]
        if subset.empty:
            latex_row.append('-')
            continue
        best = subset[detr_cols].max(axis=1).values[0]
        latex_row.append(f"{best:.3f}")

# Generate LaTeX row
row_label = "100"
latex_row_str = " & ".join([row_label] + latex_row) + " \\\\"
print(latex_row_str)
